<a href="https://colab.research.google.com/github/IlyaDenisov88/news-stock-prediction/blob/main/notebooks/extract_full_texts.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Smartlab - работает


In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from datetime import datetime, timedelta
import time
import random


HEADERS = {"User-Agent": "Mozilla/5.0"}


In [ ]:
r = requests.get('https://smart-lab.ru/news/date/2026-01-20', headers=HEADERS, timeout=10)

In [ ]:
r.text

'<!DOCTYPE html><html lang="ru"><head>\n\t<!-- Global Site Tag (gtag.js) - Google Analytics -->\n\t<script async src="https://www.googletagmanager.com/gtag/js?id=UA-16537214-3"></script>\n\t<script>\n\twindow.dataLayer = window.dataLayer || [];\n\tfunction gtag(){dataLayer.push(arguments);}\n\tgtag(\'js\', new Date());\n\tgtag(\'config\', \'UA-16537214-3\', {\n\t\t\t\'custom_map\': {\n\t\t\t\t\'dimension1\' : \'user_registred\',\n\t\t\t\t\'dimension2\' : \'content_owner\'\n\n\t\t\t},\n\n\t\t\t\'user_registred\': \'No\',\n\t\t\t\'content_owner\': \'No\'\t});\n\t</script>\n\t<meta name="push-subscribes" content="no"><title>Новости компаний и новости по акциям за дату 20.01.2026</title><meta http-equiv="content-type" content="text/html; charset=utf-8"/><link rel="manifest" href="/manifest.json"><meta name="DESCRIPTION" content="Все новости по компаниям, которые торгуются на Московской бирже. Причины роста и падения акций. Объяснение новостей экспертами. Дата 20.01.2026"/><meta name="KEYWO

In [ ]:
url = "https://smart-lab.ru/news/date/2026-01-20/"
resp = requests.get(url)
soup = BeautifulSoup(resp.text, 'html.parser')

news_items = soup.find_all('h3', class_='feed title bluid_48504')

for item in news_items:
    date = item.find('span', class_='user').text.strip()
    link = item.find('a')['href']
    title = item.find('a')['title']
    print(date, title, "https://smart-lab.ru" + link)


20/01 Трамп: пока не удается договориться, чтобы Украина и Россия одновременно проявили готовность к урегулированию конфликта https://smart-lab.ru/blog/1255548.php
20/01 📈 Акции Селигдара сегодня прибавляют 7,3% на фоне рекордных цен на золото https://smart-lab.ru/blog/1255547.php
20/01 Трамп утверждает, что лучше управляет американской экономикой, чем Уоррен Баффетт своим бизнесом https://smart-lab.ru/blog/1255545.php
20/01 Трамп: США придется вернуть полученные от пошлин доходы, если Верховный суд страны признает незаконными действия американских властей в тарифной политике https://smart-lab.ru/blog/1255541.php
20/01 📉 Dow Jones -1,72%, S&P 500 -1,93%, Nasdaq -2,18%. Американский рынок падает второй день из-за конфликта Трампа с Европой https://smart-lab.ru/blog/1255539.php
20/01 Дмитриев заявил, что его встречи в Давосе проходят конструктивно. Он также отметил, что все больше и больше людей осознают правильность позиции России https://smart-lab.ru/blog/1255538.php
20/01 ЭсЭфАй: на д

In [ ]:
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                  "AppleWebKit/537.36 (KHTML, like Gecko) "
                  "Chrome/120.0.0.0 Safari/537.36"
}

# Функция для получения новостей за одну страницу
def get_news_for_page(day_url):
    resp = requests.get(day_url, headers=headers)
    resp.raise_for_status()
    soup = BeautifulSoup(resp.text, 'html.parser')

    news_items = soup.find_all('h3', class_='feed title bluid_48504')
    news_list = []

    for item in news_items:
        date_span = item.find('span', class_='user')
        link_tag = item.find('a')

        if date_span and link_tag:
            news_date = date_span.text.strip()
            title = link_tag['title'].strip()
            link = "https://smart-lab.ru" + link_tag['href']
            news_list.append({
                'date': news_date,
                'title': title,
                'link': link
            })
    return news_list




In [ ]:
def parse_smartlab_article(link: str, headers: dict) -> dict:
    """
    Парсит smart-lab статью:
    - полное название
    - точное время
    - теги
    """
    try:
        resp = requests.get(link, headers=headers, timeout=10)
        resp.raise_for_status()
        soup = BeautifulSoup(resp.text, "html.parser")

        # Время публикации
        date_li = soup.select_one("ul.action li.date")
        published_time = None
        if date_li:
            published_time = date_li.get_text(strip=True)

        # Теги
        tags_block = soup.find("ul", class_="tags")
        tags = None
        if tags_block:
            tags = [
                a.get_text(strip=True)
                for a in tags_block.find_all("a")
            ]
            tags = ", ".join(tags)

        return {
            "published_time": published_time,
            "tags": tags
        }

    except Exception as e:
        print(f"Ошибка при парсинге статьи {link}: {e}")
        return {
            "published_time": None,
            "tags": None
        }


In [ ]:
from datetime import date
# Основной цикл по дням
start_date = date(2026, 1, 1)
end_date = date.today()

all_news = []

current_date = start_date
while current_date <= end_date:
    date_str = current_date.strftime('%Y-%m-%d')
    print(f"Сбор новостей за {date_str}...")

    for page in range(1, 6):
        time.sleep(random.uniform(0.5, 1.5))

        if page == 1:
            url = f"https://smart-lab.ru/news/date/{date_str}/"
        else:
            url = f"https://smart-lab.ru/news/date/{date_str}/page{page}/"

        try:
            news_on_page = get_news_for_page(url)
            if not news_on_page:
                break

            for article in news_on_page:
                extra = parse_smartlab_article(
                    article["link"],
                    headers=headers
                )

                record = {
                    "source": "smart-lab.ru",
                    "date_only": date_str,
                    "title": article["title"],
                    "link": article["link"],
                    "published_time": extra["published_time"],
                    "tags": extra["tags"]
                }

                all_news.append(record)

        except Exception as e:
            print(f"Ошибка при запросе {url}: {e}")
            break

    current_date += timedelta(days=1)


Сбор новостей за 2026-01-01...
Сбор новостей за 2026-01-02...
Сбор новостей за 2026-01-03...
Сбор новостей за 2026-01-04...
Сбор новостей за 2026-01-05...
Сбор новостей за 2026-01-06...
Сбор новостей за 2026-01-07...
Сбор новостей за 2026-01-08...
Сбор новостей за 2026-01-09...
Сбор новостей за 2026-01-10...
Сбор новостей за 2026-01-11...
Сбор новостей за 2026-01-12...
Сбор новостей за 2026-01-13...
Сбор новостей за 2026-01-14...
Сбор новостей за 2026-01-15...
Сбор новостей за 2026-01-16...
Сбор новостей за 2026-01-17...
Сбор новостей за 2026-01-18...
Сбор новостей за 2026-01-19...
Сбор новостей за 2026-01-20...
Сбор новостей за 2026-01-21...
Сбор новостей за 2026-01-22...
Сбор новостей за 2026-01-23...
Сбор новостей за 2026-01-24...
Сбор новостей за 2026-01-25...
Сбор новостей за 2026-01-26...
Сбор новостей за 2026-01-27...
Сбор новостей за 2026-01-28...
Сбор новостей за 2026-01-29...
Сбор новостей за 2026-01-30...


In [ ]:
# Создаем DataFrame
df = pd.DataFrame(all_news)
df[['title', 'published_time', 'tags']].head()

,title,published_time,tags
0,Вице-президент Венесуэлы Делси Родригес официа...,"05 января 2026, 22:10","Делси Родригес, Николас Мадуро, Венесуэла"
1,"Госдепартамент США официально заявил, что Запа...","05 января 2026, 21:20","США, Госдепартамент США"
2,На этой неделе США проведут встречу с руководи...,"05 января 2026, 21:03","Венесуэла, США, нефтяные компании, акции, chev..."
3,Мадуро заявил о своей невиновности в суде Нью-...,"05 января 2026, 20:20","Венесуэла, Нью-Йорк, США, Николас Мадуро, Bloo..."
4,Власти Венесуэлы объявили о мобилизации военны...,"05 января 2026, 20:09","Венесуэла, военный режим, сша, нефтяная промыш..."


In [ ]:
# При желании можно сохранить в CSV
df.to_csv('/content/drive/MyDrive/ВКР/data/news/full_data/smartlab_news_2024-2026.csv', index=False)

# Investfunds попытка №2 - работает


In [ ]:
import pandas as pd

df = pd.read_csv("/content/drive/MyDrive/ВКР/data/news/first_data/investfunds_news.csv")
df

,date,time,title,link,source
0,Сегодня,11:09,Ноябрь 2025 г.: розничные инвесторы увеличиваю...,https://investfunds.ru/news/171731/,Investfunds
1,Сегодня,10:13,Группа «Эталон» объявляет о начале сбора заяво...,https://investfunds.ru/news/171729/,NaN
2,23 января 2026,15:34,Инвестиционные тренды-2026: как выстраивать по...,https://investfunds.ru/news/171727/,NaN
3,23 января 2026,12:03,Какие результаты могут показать инвестфонды в ...,https://investfunds.ru/news/171725/,Коммерсантъ
4,22 января 2026,10:31,Венчурный фонд Т-банка инвестировал в разработ...,https://investfunds.ru/news/171723/,Ведомости
...,...,...,...,...,...
42737,04 января 2016,10:50,"Куда мчится экономика РФ: прогноз ""большой тро...",https://investfunds.ru/news/11851/,NaN
42738,04 января 2016,10:48,Индексы Dow Jones и S&P 500 по итогам года впе...,https://investfunds.ru/news/11849/,ПРАЙМ
42739,04 января 2016,10:46,Нефть растет фоне разрыва дипотношений между Э...,https://investfunds.ru/news/11847/,ПРАЙМ
42740,04 января 2016,10:41,Банк России аннулировал лицензию пенсионного ф...,https://investfunds.ru/news/5/,ПРАЙМ


In [ ]:
def parse_investfunds_article(link: str) -> dict:
    result = {
        "summary": None,
        "full_text": None
    }

    try:
        response = requests.get(link, timeout=15)
        response.raise_for_status()
    except Exception:
        return result

    soup = BeautifulSoup(response.text, "html.parser")

    # --- анонс ---
    summary_tag = soup.find("p", class_="main_descr")
    if summary_tag:
        result["summary"] = summary_tag.get_text(strip=True)

    # --- основной контент ---
    content_div = soup.find("div", class_="descr")

    if content_div:
        paragraphs = content_div.find_all("p")

        if paragraphs:
            # Вариант 1: есть <p>
            text = "\n".join(
                p.get_text(strip=True)
                for p in paragraphs
                if p.get_text(strip=True)
            )
        else:
            # Вариант 2: просто текст внутри div
            text = content_div.get_text(
                separator="\n",
                strip=True
            )

        result["full_text"] = text if text else None

    return result


In [ ]:
import time
import random

df["summary"] = None
df["full_text"] = None

for idx, row in df[:10].iterrows():
    link = row["link"]

    if pd.isna(link):
        continue

    print(f"[{idx}] {link}")

    data = parse_investfunds_article(link)

    df.at[idx, "summary"] = data["summary"]
    df.at[idx, "full_text"] = data["full_text"]

    # time.sleep(random.uniform(0.5, 1.2))  # бережно к сайту
# будет работать 8 часов...

[0] https://investfunds.ru/news/171731/
[1] https://investfunds.ru/news/171729/
[2] https://investfunds.ru/news/171727/
[3] https://investfunds.ru/news/171725/
[4] https://investfunds.ru/news/171723/
[5] https://investfunds.ru/news/171721/
[6] https://investfunds.ru/news/171719/
[7] https://investfunds.ru/news/171717/
[8] https://investfunds.ru/news/171569/
[9] https://investfunds.ru/news/171715/


In [ ]:
df

,date,time,title,link,source,summary,full_text
0,Сегодня,11:09,Ноябрь 2025 г.: розничные инвесторы увеличиваю...,https://investfunds.ru/news/171731/,Investfunds,Команда Investfunds проанализировала статистик...,Количество счетов физических лиц на розничном ...
1,Сегодня,10:13,Группа «Эталон» объявляет о начале сбора заяво...,https://investfunds.ru/news/171729/,NaN,SPO позволит компании профинансировать стратег...,"МКПАО «Эталон Груп» (далее – «компания», «Груп..."
2,23 января 2026,15:34,Инвестиционные тренды-2026: как выстраивать по...,https://investfunds.ru/news/171727/,NaN,,2026 год становится переходным: от режима «ант...
3,23 января 2026,12:03,Какие результаты могут показать инвестфонды в ...,https://investfunds.ru/news/171725/,Коммерсантъ,Минувший год оказался крайне успешным для рынк...,Смягчение денежно-кредитной политики в 2025 го...
4,22 января 2026,10:31,Венчурный фонд Т-банка инвестировал в разработ...,https://investfunds.ru/news/171723/,Ведомости,Фонд «Венчурные инвестиции 1» Т-банка для физл...,Simple Company основана в 2022 г. Андреем Черн...
...,...,...,...,...,...,...,...
42737,04 января 2016,10:50,"Куда мчится экономика РФ: прогноз ""большой тро...",https://investfunds.ru/news/11851/,NaN,None,None
42738,04 января 2016,10:48,Индексы Dow Jones и S&P 500 по итогам года впе...,https://investfunds.ru/news/11849/,ПРАЙМ,None,None
42739,04 января 2016,10:46,Нефть растет фоне разрыва дипотношений между Э...,https://investfunds.ru/news/11847/,ПРАЙМ,None,None
42740,04 января 2016,10:41,Банк России аннулировал лицензию пенсионного ф...,https://investfunds.ru/news/5/,ПРАЙМ,None,None


In [ ]:
df[['date', 'time', 'title', 'source', 'summary', 'full_text']].head(10)

,date,time,title,source,summary,full_text
0,Сегодня,11:09,Ноябрь 2025 г.: розничные инвесторы увеличиваю...,Investfunds,Команда Investfunds проанализировала статистик...,Количество счетов физических лиц на розничном ...
1,Сегодня,10:13,Группа «Эталон» объявляет о начале сбора заяво...,NaN,SPO позволит компании профинансировать стратег...,"МКПАО «Эталон Груп» (далее – «компания», «Груп..."
2,23 января 2026,15:34,Инвестиционные тренды-2026: как выстраивать по...,NaN,,2026 год становится переходным: от режима «ант...
3,23 января 2026,12:03,Какие результаты могут показать инвестфонды в ...,Коммерсантъ,Минувший год оказался крайне успешным для рынк...,Смягчение денежно-кредитной политики в 2025 го...
4,22 января 2026,10:31,Венчурный фонд Т-банка инвестировал в разработ...,Ведомости,Фонд «Венчурные инвестиции 1» Т-банка для физл...,Simple Company основана в 2022 г. Андреем Черн...
5,21 января 2026,13:28,УК ВИМ Инвестиции заработала свыше 4 млрд рубл...,NaN,,Максимальная доходность эндаументов под управл...
6,21 января 2026,09:52,В I чтении принят проект о праве компаний груп...,Интерфакс,Госдума приняла в первом чтении правительствен...,Документ (№1082205-8) вносит изменения в закон...
7,20 января 2026,16:21,Steplife привлекает 200 млн рублей в закрытом ...,Агентство Бизнес Новостей,"Российская компания Steplife, ведущий отечеств...",Компания предлагает инвесторам 100 000 обыкнов...
8,20 января 2026,12:58,Навигатор коллективных инвестиций: как меняютс...,Альфа-Капитал,"Навигатор коллективных инвестиций, рассчитывае...","Приток в фонды облигаций достиг 25,9 млрд рубл..."
9,20 января 2026,10:28,Эксперты прогнозируют IPO-оттепель,Агентство Бизнес Новостей,"На рынке IPO в 2026 году начнется оживление, о...",«Смягчение кредитно-денежной политики будет сп...


In [ ]:
len(str(df.full_text[0]))

4600

In [ ]:
# сохраняем в CSV
# df.to_csv('/content/drive/MyDrive/ВКР/data/news/full_data/investfunds_news.csv')

# Коммерсантъ

In [ ]:
def parse_kommersant_article(link: str) -> dict:
    result = {"description": None, "keywords": None, "full_text": None}

    try:
        response = requests.get(link, timeout=15)
        response.raise_for_status()
    except Exception:
        return result

    soup = BeautifulSoup(response.text, "html.parser")

    # description и keywords
    desc_meta = soup.find("meta", attrs={"name": "description"})
    if desc_meta:
        result["description"] = desc_meta.get("content")

    keywords_meta = soup.find("meta", attrs={"name": "keywords"})
    if keywords_meta:
        result["keywords"] = keywords_meta.get("content")

    return result



In [ ]:
df = pd.read_csv("/content/drive/MyDrive/ВКР/data/news/first_data/kommersant_news.csv", delimiter=';')
df = df[:10]  # если хочешь обрезать для теста

df[["description", "keywords"]] = (
    df["link"]
    .apply(parse_kommersant_article)
    .apply(pd.Series)
)

df.head()


,rubric,date,time,title,link,description,keywords,full_text
0,Экономика,2026-01-27,01:03,Доллар не теряет к себе внимания,https://www.kommersant.ru/doc/8378451,Банк международных расчетов (BIS) проанализиро...,"международных,расчетов,проанализировал,валютну...",None
1,Экономика,2026-01-27,00:00,«Единому заказчику» не хватило эффективности,https://www.kommersant.ru/doc/8378420,Счетная палата РФ выявила недостаточную эффект...,"Счетная,палата,сочла,деятельность,публично-пра...",None
2,Экономика,2026-01-26,19:45,ЦБ сообщил о повышенных инфляционных и ценовых...,https://www.kommersant.ru/doc/8378425,Банк России зафиксировал рост ценовых ожиданий...,"сообщил,повышенных,инфляционных,ценовых,ожидан...",None
3,Экономика,2026-01-26,15:55,Новые правила деревянных вложений,https://www.kommersant.ru/doc/8377581,Минпромторг представил проект постановления о ...,"Минпромторг,намерен,добиться,повышения,эффекти...",None
4,Экономика,2026-01-26,04:20,«Известия»: количество открытых компаний в 202...,https://www.kommersant.ru/doc/8377650,В 2025 году в России зарегистрировано 173 тыс....,"«Известия»:,количество,открытых,компаний,стало...",None


In [ ]:
len(str(df.description[1]))

383

In [ ]:
# сохраняем в CSV (UTF-8, чтобы кириллица не слетела)
# df.to_csv("/content/drive/MyDrive/ВКР/data/news/kommersant_news.csv", index=False, sep=";")


# Interfax

In [ ]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
from time import sleep

def parse_interfax_article(link: str) -> dict:
    result = {"description": None, "tags": [], "full_text": None}
    try:
        response = requests.get(link, timeout=15)
        response.raise_for_status()
    except Exception:
        return result

    soup = BeautifulSoup(response.text, "html.parser")

    # description
    desc_meta = soup.find("meta", attrs={"name": "description"})
    if desc_meta:
        result["description"] = desc_meta.get("content")

    # полный текст статьи
    article = soup.find("article", itemprop="articleBody")
    if article:
        paragraphs = article.find_all("p")
        result["full_text"] = "\n\n".join(p.get_text(strip=True) for p in paragraphs)

    # теги
    tags_div = soup.find("div", class_="textMTags")
    if tags_div:
        result["tags"] = [a.get_text(strip=True) for a in tags_div.find_all("a")]

    return result



In [ ]:
df = pd.read_csv("/content/drive/MyDrive/ВКР/data/news/first_data/interfax_news.csv")
df

,site,date,time,title,link,description
0,interfax.ru,2026-01-27,12:06,"Опрос ЦБ показал, что инфляционные ожидания на...",https://www.interfax.ru/business/1069659,NaN
1,interfax.ru,2026-01-27,10:09,Рубль утром дешевеет в паре с юанем на фоне не...,https://www.interfax.ru/business/1069640,NaN
2,interfax.ru,2026-01-27,10:08,Рынок акций открылся в основную сессию незначи...,https://www.interfax.ru/business/1069639,NaN
3,interfax.ru,2026-01-27,09:50,Китайская Anta Sports покупает 29% акций Puma ...,https://www.interfax.ru/business/1069634,NaN
4,interfax.ru,2026-01-27,09:44,"""Ъ"" сообщил о предложении Минфина легализовать...",https://www.interfax.ru/business/1069630,NaN
...,...,...,...,...,...,...
21593,interfax.ru,2024-11-25,08:42,"Нефть Brent подешевела до $74,74 за баррель",https://www.interfax.ru/business/994253,NaN
21594,interfax.ru,2024-11-25,08:13,ЦБ Китая сохранил ставку MLF на уровне 2%,https://www.interfax.ru/world/994251,NaN
21595,interfax.ru,2024-11-25,07:35,Фондовые индексы США выросли в пятницу и по ит...,https://www.interfax.ru/business/994244,NaN
21596,interfax.ru,2024-11-25,06:03,"Европейские акции уверенно выросли в пятницу, ...",https://www.interfax.ru/business/994239,NaN


In [ ]:
# Загружаем CSV
df = pd.read_csv("/content/drive/MyDrive/ВКР/data/news/first_data/interfax_news.csv")

df = df[:10]

# Добавляем новые столбцы
df["description"] = None
df["full_text"] = None
df["tags"] = None

# Проходим по всем ссылкам
for idx, link in enumerate(df["link"]):
    print(f"Обрабатываем {idx+1}/{len(df)}: {link}")
    data = parse_interfax_article(link)
    df.at[idx, "description"] = data["description"]
    df.at[idx, "full_text"] = data["full_text"]
    df.at[idx, "tags"] = ", ".join(data["tags"]) if data["tags"] else None
    sleep(1)  # пауза, чтобы не перегружать сервер




Обрабатываем 1/10: https://www.interfax.ru/business/1069659
Обрабатываем 2/10: https://www.interfax.ru/business/1069640
Обрабатываем 3/10: https://www.interfax.ru/business/1069639
Обрабатываем 4/10: https://www.interfax.ru/business/1069634
Обрабатываем 5/10: https://www.interfax.ru/business/1069630
Обрабатываем 6/10: https://www.interfax.ru/business/1069620
Обрабатываем 7/10: https://www.interfax.ru/business/1069612
Обрабатываем 8/10: https://www.interfax.ru/world/1069608
Обрабатываем 9/10: https://www.interfax.ru/world/1069607
Обрабатываем 10/10: https://www.interfax.ru/world/1069595


In [ ]:
df.head()

,site,date,time,title,link,description,full_text,tags
0,interfax.ru,2026-01-27,12:06,"Опрос ЦБ показал, что инфляционные ожидания на...",https://www.interfax.ru/business/1069659,Интерфакс: Инфляционные ожидания населения РФ ...,Москва. 27 января. INTERFAX.RU - Инфляционные ...,"ЦБ РФ, Банк России"
1,interfax.ru,2026-01-27,10:09,Рубль утром дешевеет в паре с юанем на фоне не...,https://www.interfax.ru/business/1069640,Интерфакс: Китайский юань растет на Московской...,Москва. 27 января. INTERFAX.RU - Китайский юан...,"Китай, Московская биржа, Brent, WTI"
2,interfax.ru,2026-01-27,10:08,Рынок акций открылся в основную сессию незначи...,https://www.interfax.ru/business/1069639,Интерфакс: Рынок акций РФ на старте торгов осн...,Москва. 27 января. INTERFAX.RU - Рынок акций Р...,Московская биржа
3,interfax.ru,2026-01-27,09:50,Китайская Anta Sports покупает 29% акций Puma ...,https://www.interfax.ru/business/1069634,Интерфакс: Китайский производитель спортивных ...,Москва. 27 января. INTERFAX.RU - Китайский пр...,"Puma, Anta"
4,interfax.ru,2026-01-27,09:44,"""Ъ"" сообщил о предложении Минфина легализовать...",https://www.interfax.ru/business/1069630,Интерфакс: Министр финансов Антон Силуанов пре...,Москва. 27 января. INTERFAX.RU - Министр финан...,"Антон Силуанов, Владимир Путин, Минфин РФ"


In [ ]:
len(str(df.full_text[0]))

862

In [ ]:
# сохраняем
# df.to_csv("/content/drive/MyDrive/ВКР/data/news/full_data/interfax_news_full.csv", index=False)


# Comnews

In [ ]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
from time import sleep

def parse_comnews_article(link: str) -> dict:
    result = {"description": None, "full_text": None, "author": None, "tags": []}
    try:
        response = requests.get(link, timeout=15)
        response.raise_for_status()
    except Exception:
        return result

    soup = BeautifulSoup(response.text, "html.parser")

    # description
    desc_meta = soup.find("meta", attrs={"name": "description"})
    if desc_meta:
        result["description"] = desc_meta.get("content")

    # полный текст статьи
    body_div = soup.find("div", class_="field-name-body")
    if body_div:
        paragraphs = body_div.find_all("p")
        result["full_text"] = "\n\n".join(p.get_text(strip=True) for p in paragraphs)

    # автор
    author_div = soup.find("div", class_="field-name-authors")
    if author_div:
        author_span = author_div.find("span")
        if author_span:
            result["author"] = author_span.get_text(strip=True).replace("\n", " ")


    return result




In [ ]:
df = pd.read_csv("/content/drive/MyDrive/ВКР/data/news/first_data/comnews_news.csv")
df

,date,title,link,description
0,27.01.2026,"""Софтлайн"" выпустит облигации на 3 млрд руб.",https://www.comnews.ru/content/243439/2026-01-...,"ПАО ""Софтлайн"" планирует выпустить биржевые об..."
1,26.01.2026,Криптопаузе быть… или не быть,https://www.comnews.ru/content/243421/2026-01-...,ЛДПР предложила предоставить действующим участ...
2,26.01.2026,WAF и Anti-DDoS в России ускоряют темпы роста,https://www.comnews.ru/content/243419/2026-01-...,В 2025 г. объем потребления WAF и Anti-DDoS в ...
3,23.01.2026,"Сбер приобрел долю в ПАО ""Элемент""",https://www.comnews.ru/content/243404/2026-01-...,"Сбер закрыл сделку по покупке 41,9% акций в ПА..."
4,23.01.2026,"Процессная аналитика дала ""Сберу"" финансовый э...",https://www.comnews.ru/content/243391/2026-01-...,Финансовый эффект от процессной аналитики (Pro...
...,...,...,...,...
2200,16.09.2019,ИКТ продолжает лидировать,https://www.comnews.ru/content/122013/2019-09-...,Российский рынок венчурных сделок в первом пол...
2201,12.09.2019,"Ростех и ""Дигинавис"" создадут систему организа...",https://www.comnews.ru/content/121968/2019-09-...,"Госкорпорация Ростех и российская компания ""Ди..."
2202,12.09.2019,Венчурный фонд Росинфокоминвест сможет предост...,https://www.comnews.ru/content/121967/2019-09-...,"Государственный венчурный фонд ""Росинфокоминве..."
2203,11.09.2019,"""Ростелеком"" продал ""Центральный телеграф""",https://www.comnews.ru/content/121948/2019-09-...,"Помещения, принадлежащие ПАО ""Центральный теле..."


In [ ]:
# Загружаем CSV
df = pd.read_csv("/content/drive/MyDrive/ВКР/data/news/first_data/comnews_news.csv")

df = df[:10]

# Добавляем новые столбцы
df["full_text"] = None
df["author"] = None

# Проходим по всем ссылкам
for idx, link in enumerate(df["link"]):
    print(f"Обрабатываем {idx+1}/{len(df)}: {link}")
    data = parse_comnews_article(link)
    df.at[idx, "full_text"] = data["full_text"]
    df.at[idx, "author"] = data["author"]
    sleep(1)  # пауза между запросами



Обрабатываем 1/10: https://www.comnews.ru/content/243439/2026-01-27/2026-w05/1008/softlayn-vypustit-obligacii-3-mlrd-rub
Обрабатываем 2/10: https://www.comnews.ru/content/243421/2026-01-26/2026-w05/1008/kriptopauze-byt-ili-ne-byt
Обрабатываем 3/10: https://www.comnews.ru/content/243419/2026-01-26/2026-w05/1008/waf-i-anti-ddos-rossii-uskoryayut-tempy-rosta
Обрабатываем 4/10: https://www.comnews.ru/content/243404/2026-01-23/2026-w04/1010/sber-priobrel-dolyu-pao-element
Обрабатываем 5/10: https://www.comnews.ru/content/243391/2026-01-23/2026-w04/1008/processnaya-analitika-dala-sberu-finansovyy-effekt-neskolko-milliardov-rubley
Обрабатываем 6/10: https://www.comnews.ru/content/243378/2026-01-22/2026-w04/1008/apk-i-reteyl-uglubilis-it
Обрабатываем 7/10: https://www.comnews.ru/content/243362/2026-01-21/2026-w04/1010/marketologi-zhdut-poyavleniya-novykh-rossiyskikh-reklamnykh-platform-2026-g
Обрабатываем 8/10: https://www.comnews.ru/content/243361/2026-01-21/2026-w04/1010/marketpleys-mvideo-z

In [ ]:
df.head()

,date,title,link,description,full_text,author
0,27.01.2026,"""Софтлайн"" выпустит облигации на 3 млрд руб.",https://www.comnews.ru/content/243439/2026-01-...,"ПАО ""Софтлайн"" планирует выпустить биржевые об...","ПАО ""Софтлайн"" планирует разместить выпуск бир...",ПавелКоролев
1,26.01.2026,Криптопаузе быть… или не быть,https://www.comnews.ru/content/243421/2026-01-...,ЛДПР предложила предоставить действующим участ...,Об этом 22 января заявил председатель ЛДПР Лео...,АлексейМиколенко
2,26.01.2026,WAF и Anti-DDoS в России ускоряют темпы роста,https://www.comnews.ru/content/243419/2026-01-...,В 2025 г. объем потребления WAF и Anti-DDoS в ...,Об этом ComNews рассказал соос­но­ватель ана­л...,АннаШвецова
3,23.01.2026,"Сбер приобрел долю в ПАО ""Элемент""",https://www.comnews.ru/content/243404/2026-01-...,"Сбер закрыл сделку по покупке 41,9% акций в ПА...","Сбер закрыл сделку по покупке 41,9% акций в ПА...",None
4,23.01.2026,"Процессная аналитика дала ""Сберу"" финансовый э...",https://www.comnews.ru/content/243391/2026-01-...,Финансовый эффект от процессной аналитики (Pro...,"Заместитель председателя правления ПАО ""Сберба...",ЮлияСтрелец


In [ ]:
df.tags.value_counts()

,count
tags,
"Связь и ТВ, Беспроводная связь, Фиксированная связь, Спутниковая связь, Почтовая связь, Вещание, Контент, Технологии, Искусственный интеллект и сквозные технологии, Оборудование, ПО, Электроника, Сервисы, Дата-центры, Cloud, Электронная коммерция, Информационная безопасность, Интеграция, Регулирование, Нормативные акты, Суды, Госполитика, Санкции, Импортозамещение, Инвестиции, M&A, Отчеты, Криптовалюта, Конкурсы / тендеры, Ритейл, Кадры, Мобилизация для ИТ, РФ, Москва, Санкт-Петербург, ЦФО, СЗФО, ДФО, СФО, УрФО, ПФО, СКФО, ЮФО, В мире, Отчеты, Конкурсы / тендеры, Связь и ТВ, Кадры, Вакансии в IT, Мобилизация для ИТ, Сервисы, Технологии, Санкции, Импортозамещение, Регулирование, Инвестиции, Ритейл, Регионы",1
"Связь и ТВ, Беспроводная связь, Фиксированная связь, Спутниковая связь, Почтовая связь, Вещание, Контент, Технологии, Искусственный интеллект и сквозные технологии, Оборудование, ПО, Электроника, Сервисы, Дата-центры, Cloud, Электронная коммерция, Информационная безопасность, Интеграция, Регулирование, Нормативные акты, Суды, Госполитика, Санкции, Импортозамещение, Инвестиции, M&A, Отчеты, Криптовалюта, Конкурсы / тендеры, Ритейл, Кадры, Мобилизация для ИТ, РФ, Москва, Санкт-Петербург, ЦФО, СЗФО, ДФО, СФО, УрФО, ПФО, СКФО, ЮФО, В мире, Оборудование, Госполитика, Криптовалюта, Связь и ТВ, Кадры, Вакансии в IT, Мобилизация для ИТ, Сервисы, Технологии, Санкции, Импортозамещение, Регулирование, Инвестиции, Ритейл, Регионы",1
"Связь и ТВ, Беспроводная связь, Фиксированная связь, Спутниковая связь, Почтовая связь, Вещание, Контент, Технологии, Искусственный интеллект и сквозные технологии, Оборудование, ПО, Электроника, Сервисы, Дата-центры, Cloud, Электронная коммерция, Информационная безопасность, Интеграция, Регулирование, Нормативные акты, Суды, Госполитика, Санкции, Импортозамещение, Инвестиции, M&A, Отчеты, Криптовалюта, Конкурсы / тендеры, Ритейл, Кадры, Мобилизация для ИТ, РФ, Москва, Санкт-Петербург, ЦФО, СЗФО, ДФО, СФО, УрФО, ПФО, СКФО, ЮФО, В мире, Информационная безопасность, Отчеты, Связь и ТВ, Кадры, Вакансии в IT, Мобилизация для ИТ, Сервисы, Технологии, Санкции, Импортозамещение, Регулирование, Инвестиции, Ритейл, Регионы",1
"Связь и ТВ, Беспроводная связь, Фиксированная связь, Спутниковая связь, Почтовая связь, Вещание, Контент, Технологии, Искусственный интеллект и сквозные технологии, Оборудование, ПО, Электроника, Сервисы, Дата-центры, Cloud, Электронная коммерция, Информационная безопасность, Интеграция, Регулирование, Нормативные акты, Суды, Госполитика, Санкции, Импортозамещение, Инвестиции, M&A, Отчеты, Криптовалюта, Конкурсы / тендеры, Ритейл, Кадры, Мобилизация для ИТ, РФ, Москва, Санкт-Петербург, ЦФО, СЗФО, ДФО, СФО, УрФО, ПФО, СКФО, ЮФО, В мире, M&A, Связь и ТВ, Кадры, Вакансии в IT, Мобилизация для ИТ, Сервисы, Технологии, Санкции, Импортозамещение, Регулирование, Инвестиции, Ритейл, Регионы",1
"Связь и ТВ, Беспроводная связь, Фиксированная связь, Спутниковая связь, Почтовая связь, Вещание, Контент, Технологии, Искусственный интеллект и сквозные технологии, Оборудование, ПО, Электроника, Сервисы, Дата-центры, Cloud, Электронная коммерция, Информационная безопасность, Интеграция, Регулирование, Нормативные акты, Суды, Госполитика, Санкции, Импортозамещение, Инвестиции, M&A, Отчеты, Криптовалюта, Конкурсы / тендеры, Ритейл, Кадры, Мобилизация для ИТ, РФ, Москва, Санкт-Петербург, ЦФО, СЗФО, ДФО, СФО, УрФО, ПФО, СКФО, ЮФО, В мире, Искусственный интеллект и сквозные технологии, Отчеты, Связь и ТВ, Кадры, Вакансии в IT, Мобилизация для ИТ, Сервисы, Технологии, Санкции, Импортозамещение, Регулирование, Инвестиции, Ритейл, Регионы",1
"Связь и ТВ, Беспроводная связь, Фиксированная связь, Спутниковая связь, Почтовая связь, Вещание, Контент, Технологии, Искусственный интеллект и сквозные технологии, Оборудование, ПО, Электроника, Сервисы, Дата-центры, Cloud, Электронная коммерция, Информационная безопасность, Интеграция, Регулирование, Нормативные акты, Суды, Госполитика, Санкции, Импортозамещени

In [ ]:
len(str(df.full_text[0]))

3895

In [ ]:
# Сохраняем в CSV
# df.to_csv("/content/drive/MyDrive/ВКР/data/news/full_data/comnews_news_full.csv", index=False)